# Identifiability Conditions for Neural Networks

## Background: Fefferman's Framework

**Identifiability** asks a fundamental question: if two neural networks produce the same input-output mapping, must their parameters be related by a known set of transformations?

Charles Fefferman's seminal 1994 paper *"Reconstructing a Neural Net from its Output"* established that, under certain regularity conditions, the parameters of a feedforward neural network **can** be recovered (up to symmetries) from its input-output map. The key conditions are:

1. **No-Clones Condition**: No two neurons in the same hidden layer share identical incoming weights and biases.
2. **Non-Degeneracy**: Every hidden neuron contributes to the network output (no dead or redundant neurons).
3. **Self-Avoiding Property**: Weight and bias configurations avoid degenerate overlaps (distinct bias magnitudes, non-trivial weight ratios).

When these conditions hold and the activation function is **analytic** (e.g., sigmoid, tanh), the network is identifiable up to:
- **Neuron permutations** within each hidden layer
- **Sign flips** (for odd activations like tanh: $\tanh(-x) = -\tanh(x)$)

### References
- Fefferman, C. (1994). *Reconstructing a Neural Net from its Output*. Revista Matematica Iberoamericana.
- Bona-Pellissier, Miche, Malgouyres (2022). *Parameter Identifiability of Neural Networks with ReLU, Tanh, and Sigmoid Activations*.
- Petzka, H., Trimmel, M. (2020). *On the Identifiability of Neural Networks*.

---

In this notebook, we implement and verify each condition on concrete PyTorch networks.

## 1. Imports and Setup

In [ ]:
import sys
import os
import copy
from itertools import combinations

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# Add src/ to path for project module imports
sys.path.insert(0, '..')
from src.identifiability_checks import (
    build_network,
    extract_parameters,
    check_no_clones,
    check_non_degeneracy,
    check_self_avoiding,
    check_fefferman_assumptions,
    identifiability_report,
)
from src.activation_analysis import analyze_activation_properties
from src.visualization import (
    plot_network_architecture,
    plot_neuron_similarity_matrix,
)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version:   {np.__version__}")

## 2. Building Sample Feedforward Networks

We construct three networks with the **same architecture** `[5, 10, 10, 1]` but different activation functions to examine how the choice of activation impacts identifiability.

In [ ]:
architecture = [5, 10, 10, 1]

# Build networks with different activations
torch.manual_seed(42)
model_sigmoid = build_network(architecture, activation='sigmoid')

torch.manual_seed(42)
model_tanh = build_network(architecture, activation='tanh')

torch.manual_seed(42)
model_relu = build_network(architecture, activation='relu')

models = {
    'sigmoid': model_sigmoid,
    'tanh': model_tanh,
    'relu': model_relu,
}

for name, model in models.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name:>8}: {n_params:,} parameters")
    print(f"          {model}")
    print()

In [ ]:
# Quick sanity check: forward pass
X_test = torch.randn(5, architecture[0])

for name, model in models.items():
    model.eval()
    with torch.no_grad():
        y = model(X_test)
    print(f"{name:>8} output shape: {y.shape}, sample: {y[:3].squeeze().tolist()}")

## 3. No-Clones Condition

### Definition
Two neurons $j$ and $j'$ in the same hidden layer $l$ are **clones** if they have identical incoming weight vectors and biases:
$$W^{(l)}_{j,:} = W^{(l)}_{j',:} \quad \text{and} \quad b^{(l)}_j = b^{(l)}_{j'}$$

Clone pairs break identifiability because swapping or merging clones yields a different parameterization with the same input-output map.

### Experiment
We first check a randomly initialized network (which should be clone-free), then **manually inject** clone neurons to verify our detection works.

In [ ]:
# Check no-clones on a standard randomly-initialized network
torch.manual_seed(42)
model_clean = build_network([5, 10, 10, 1], 'tanh')
layers_clean = extract_parameters(model_clean)

clone_result_clean = check_no_clones(layers_clean)
print("=== Clone Check on Randomly Initialized Network ===")
print(f"Is clone-free: {clone_result_clean['is_clone_free']}")
for info in clone_result_clean['layers']:
    print(f"  Layer {info['layer_index']}: {info['num_neurons']} neurons, "
          f"clone pairs: {info['clone_pairs']}, clone-free: {info['is_clone_free']}")

In [ ]:
# Now inject clone neurons: copy neuron 0 -> neuron 1 in the first hidden layer
model_cloned = copy.deepcopy(model_clean)
linear_layers = [m for m in model_cloned.modules() if isinstance(m, nn.Linear)]

with torch.no_grad():
    # Make neuron 1 a clone of neuron 0 in the first hidden layer
    linear_layers[0].weight[1] = linear_layers[0].weight[0].clone()
    linear_layers[0].bias[1] = linear_layers[0].bias[0].clone()
    
    # Also make neurons 5 and 6 clones of each other
    linear_layers[0].weight[6] = linear_layers[0].weight[5].clone()
    linear_layers[0].bias[6] = linear_layers[0].bias[5].clone()

layers_cloned = extract_parameters(model_cloned)
clone_result_cloned = check_no_clones(layers_cloned)

print("=== Clone Check After Injecting Clone Neurons ===")
print(f"Is clone-free: {clone_result_cloned['is_clone_free']}")
for info in clone_result_cloned['layers']:
    print(f"  Layer {info['layer_index']}: {info['num_neurons']} neurons, "
          f"clone pairs: {info['clone_pairs']}, clone-free: {info['is_clone_free']}")

In [ ]:
# Visualize neuron similarity to see clones as high-similarity entries
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (title, layers) in zip(axes, [
    ('Clean Network', layers_clean),
    ('Network with Clones', layers_cloned)
]):
    W = layers[0]['weight']  # First hidden layer
    W_norm = W / (W.norm(dim=1, keepdim=True) + 1e-8)
    similarity = (W_norm @ W_norm.T).numpy()
    
    im = ax.imshow(similarity, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(f'{title}\nLayer 0 Cosine Similarity', fontsize=12)
    ax.set_xlabel('Neuron Index')
    ax.set_ylabel('Neuron Index')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Clone Detection via Neuron Similarity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Non-Degeneracy Check

### Definition
A network is **non-degenerate** if every hidden neuron contributes to the network's output. Formally, for each hidden neuron $j$ in layer $l$, there exists at least one input $x$ such that zeroing out neuron $j$ changes the output.

A **dead neuron** is one whose removal does not affect the output for any input. Dead neurons create ambiguity in the parameterization.

### Experiment
We test a normal network, then create one with deliberately dead neurons (by zeroing outgoing weights).

In [ ]:
# Non-degeneracy check on clean network
torch.manual_seed(42)
model_nd = build_network([5, 10, 10, 1], 'tanh')

nd_result_clean = check_non_degeneracy(model_nd, [5, 10, 10, 1], n_samples=500)
print("=== Non-Degeneracy Check (Clean Network) ===")
print(f"Is non-degenerate: {nd_result_clean['is_non_degenerate']}")
for info in nd_result_clean['layers']:
    print(f"  Layer {info['layer_index']}: {info['num_neurons']} neurons, "
          f"dead neurons: {info['dead_neurons']}")

In [ ]:
# Create a degenerate network: zero out outgoing weights of neurons 2 and 7
model_degenerate = copy.deepcopy(model_nd)
linear_layers_deg = [m for m in model_degenerate.modules() if isinstance(m, nn.Linear)]

with torch.no_grad():
    # Zero outgoing connections of neuron 2 in layer 0 -> makes it dead
    linear_layers_deg[1].weight[:, 2] = 0.0
    # Zero outgoing connections of neuron 7 in layer 0
    linear_layers_deg[1].weight[:, 7] = 0.0

nd_result_deg = check_non_degeneracy(model_degenerate, [5, 10, 10, 1], n_samples=500)
print("=== Non-Degeneracy Check (Degenerate Network) ===")
print(f"Is non-degenerate: {nd_result_deg['is_non_degenerate']}")
for info in nd_result_deg['layers']:
    status = 'OK' if info['is_non_degenerate'] else f"DEAD: {info['dead_neurons']}"
    print(f"  Layer {info['layer_index']}: {info['num_neurons']} neurons -> {status}")

In [ ]:
# Visualize: output difference when each neuron in layer 0 is removed
model_nd.eval()
X_probe = torch.randn(200, 5)

with torch.no_grad():
    y_base = model_nd(X_probe)

contributions_clean = []
contributions_deg = []

for j in range(10):  # 10 neurons in layer 0
    # Clean model
    m_tmp = copy.deepcopy(model_nd)
    ll = [m for m in m_tmp.modules() if isinstance(m, nn.Linear)]
    with torch.no_grad():
        ll[1].weight[:, j] = 0.0
        y_mod = m_tmp(X_probe)
    contributions_clean.append((y_base - y_mod).abs().mean().item())
    
    # Degenerate model
    m_tmp2 = copy.deepcopy(model_degenerate)
    ll2 = [m for m in m_tmp2.modules() if isinstance(m, nn.Linear)]
    with torch.no_grad():
        y_base_deg = model_degenerate(X_probe)
        ll2[1].weight[:, j] = 0.0
        y_mod2 = m_tmp2(X_probe)
    contributions_deg.append((y_base_deg - y_mod2).abs().mean().item())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_clean = ['#2ecc71' if c > 1e-6 else '#e74c3c' for c in contributions_clean]
axes[0].bar(range(10), contributions_clean, color=colors_clean, edgecolor='black', alpha=0.8)
axes[0].set_title('Clean Network: Neuron Contributions (Layer 0)', fontsize=12)
axes[0].set_xlabel('Neuron Index')
axes[0].set_ylabel('Mean |Output Change|')
axes[0].axhline(1e-6, color='red', linestyle='--', alpha=0.5, label='Dead threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

colors_deg = ['#2ecc71' if c > 1e-6 else '#e74c3c' for c in contributions_deg]
axes[1].bar(range(10), contributions_deg, color=colors_deg, edgecolor='black', alpha=0.8)
axes[1].set_title('Degenerate Network: Neuron Contributions (Layer 0)', fontsize=12)
axes[1].set_xlabel('Neuron Index')
axes[1].set_ylabel('Mean |Output Change|')
axes[1].axhline(1e-6, color='red', linestyle='--', alpha=0.5, label='Dead threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Non-Degeneracy: Neuron Contribution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Self-Avoiding Property

### Definition
The self-avoiding property requires that weight/bias configurations do not create degenerate overlaps. Specifically:

1. **Distinct bias magnitudes**: For neurons $j \neq j'$ in the same layer, $|b_j| \neq |b_{j'}|$.
2. **Non-trivial weight ratios**: The ratio $W_{j,k} / W_{j',k}$ should not be a simple fraction $p/q$ for small integers.

These conditions ensure that the network's parameter space is "generic" in the sense that the set of non-identifiable networks has measure zero.

In [ ]:
# Self-avoiding check on clean network
torch.manual_seed(42)
model_sa = build_network([5, 10, 10, 1], 'tanh')
layers_sa = extract_parameters(model_sa)

sa_result = check_self_avoiding(layers_sa)
print("=== Self-Avoiding Property (Random Init) ===")
print(f"Is self-avoiding: {sa_result['is_self_avoiding']}")
for info in sa_result['layers']:
    n_violations = len(info['violations'])
    bias_violations = len([v for v in info['violations'] if v['type'] == 'bias_magnitude_collision'])
    ratio_violations = len([v for v in info['violations'] if v['type'] == 'simple_fraction_ratio'])
    print(f"  Layer {info['layer_index']}: {info['num_neurons']} neurons")
    print(f"    Bias magnitude collisions: {bias_violations}")
    print(f"    Simple fraction ratios:    {ratio_violations}")
    print(f"    Self-avoiding: {info['is_self_avoiding']}")

In [ ]:
# Create a network that violates self-avoiding: set equal bias magnitudes
model_sa_viol = copy.deepcopy(model_sa)
ll_sa = [m for m in model_sa_viol.modules() if isinstance(m, nn.Linear)]

with torch.no_grad():
    # Set biases of neurons 0 and 1 to have the same magnitude
    ll_sa[0].bias[0] = torch.tensor(0.5)
    ll_sa[0].bias[1] = torch.tensor(-0.5)  # |b_0| = |b_1| = 0.5
    # Set biases of neurons 3 and 4 to be identical
    ll_sa[0].bias[3] = torch.tensor(0.3)
    ll_sa[0].bias[4] = torch.tensor(0.3)

layers_sa_viol = extract_parameters(model_sa_viol)
sa_result_viol = check_self_avoiding(layers_sa_viol)

print("=== Self-Avoiding Property (Violated) ===")
print(f"Is self-avoiding: {sa_result_viol['is_self_avoiding']}")
for info in sa_result_viol['layers']:
    bias_violations = [v for v in info['violations'] if v['type'] == 'bias_magnitude_collision']
    if bias_violations:
        print(f"  Layer {info['layer_index']} - Bias magnitude collisions:")
        for v in bias_violations:
            print(f"    Neurons {v['neurons']}: |b| = {v['values'][0]:.4f}, {v['values'][1]:.4f}")

In [ ]:
# Visualize bias magnitude distributions for both networks
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, layers) in zip(axes, [
    ('Self-Avoiding (Clean)', layers_sa),
    ('Self-Avoiding Violated', layers_sa_viol)
]):
    biases = layers[0]['bias'].numpy()
    bias_mags = np.abs(biases)
    
    colors = plt.cm.Set2(np.linspace(0, 1, len(biases)))
    ax.bar(range(len(biases)), bias_mags, color=colors, edgecolor='black', alpha=0.8)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Neuron Index')
    ax.set_ylabel('|Bias|')
    ax.grid(True, alpha=0.3)
    
    # Mark collisions
    for i in range(len(bias_mags)):
        for j in range(i + 1, len(bias_mags)):
            if abs(bias_mags[i] - bias_mags[j]) < 1e-6:
                ax.annotate('', xy=(j, bias_mags[j] + 0.02),
                           xytext=(i, bias_mags[i] + 0.02),
                           arrowprops=dict(arrowstyle='<->', color='red', lw=2))

plt.suptitle('Bias Magnitude Analysis (Self-Avoiding Property)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Activation Function Analysis

Identifiability guarantees depend critically on the activation function:

| Property | Sigmoid | Tanh | ReLU |
|----------|---------|------|------|
| Analytic | Yes | Yes | No |
| Monotonic | Yes | Yes | Yes |
| Bounded | Yes | Yes | No |
| Odd symmetry | No | Yes ($\tanh(-x) = -\tanh(x)$) | No |
| Identifiable | Yes (perm) | Yes (perm + sign) | No (fails genericity) |

**Sigmoid** networks are identifiable up to neuron permutations. **Tanh** networks have an additional sign-flip symmetry. **ReLU** networks fail Fefferman's genericity conditions because ReLU is not analytic.

In [ ]:
# Plot activation functions and their key properties
x = torch.linspace(-5, 5, 1000)

activations_data = {
    'Sigmoid': {'fn': torch.sigmoid, 'color': '#3498db', 'identifiable': True},
    'Tanh':    {'fn': torch.tanh,    'color': '#e74c3c', 'identifiable': True},
    'ReLU':    {'fn': torch.relu,    'color': '#2ecc71', 'identifiable': False},
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for col, (name, info) in enumerate(activations_data.items()):
    y = info['fn'](x).numpy()
    color = info['color']
    
    # Activation function
    axes[0, col].plot(x.numpy(), y, color=color, linewidth=2.5)
    axes[0, col].axhline(0, color='gray', linewidth=0.5, linestyle='--')
    axes[0, col].axvline(0, color='gray', linewidth=0.5, linestyle='--')
    axes[0, col].set_title(f'{name}', fontsize=14, fontweight='bold')
    axes[0, col].set_xlabel('x')
    axes[0, col].set_ylabel('f(x)')
    axes[0, col].grid(True, alpha=0.3)
    
    # Identifiability badge
    badge_color = '#2ecc71' if info['identifiable'] else '#e74c3c'
    axes[0, col].text(0.02, 0.98, f"Identifiable: {'Yes' if info['identifiable'] else 'No'}",
                      transform=axes[0, col].transAxes, fontsize=10,
                      verticalalignment='top',
                      bbox=dict(boxstyle='round', facecolor=badge_color, alpha=0.3))
    
    # Derivative
    x_grad = x.clone().requires_grad_(True)
    y_grad = info['fn'](x_grad)
    dy = torch.autograd.grad(y_grad.sum(), x_grad)[0]
    axes[1, col].plot(x.numpy(), dy.detach().numpy(), color=color, linewidth=2.5)
    axes[1, col].axhline(0, color='gray', linewidth=0.5, linestyle='--')
    axes[1, col].axvline(0, color='gray', linewidth=0.5, linestyle='--')
    axes[1, col].set_title(f'{name} Derivative', fontsize=14)
    axes[1, col].set_xlabel('x')
    axes[1, col].set_ylabel("f'(x)")
    axes[1, col].grid(True, alpha=0.3)

plt.suptitle('Activation Functions and Their Derivatives\n'
             '(Identifiability depends on analyticity and symmetry properties)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate tanh sign-flip invariance: tanh(-x) = -tanh(x)
torch.manual_seed(42)
model_tanh_demo = build_network([3, 5, 1], 'tanh')
model_tanh_demo.eval()

X_demo = torch.randn(200, 3)

with torch.no_grad():
    y_original = model_tanh_demo(X_demo)

# Flip signs of neuron 0 in hidden layer
model_flipped = copy.deepcopy(model_tanh_demo)
ll_flip = [m for m in model_flipped.modules() if isinstance(m, nn.Linear)]

with torch.no_grad():
    # Flip incoming: negate row 0 of W^1 and b^1[0]
    ll_flip[0].weight[0] *= -1
    ll_flip[0].bias[0] *= -1
    # Flip outgoing: negate column 0 of W^2
    ll_flip[1].weight[:, 0] *= -1
    
    y_flipped = model_flipped(X_demo)

max_diff = (y_original - y_flipped).abs().max().item()
print(f"Tanh Sign-Flip Invariance Demonstration")
print(f"  Max output difference after sign flip of neuron 0: {max_diff:.2e}")
print(f"  Invariance verified: {max_diff < 1e-6}")
print()
print("This confirms that tanh networks have sign-flip symmetry:")
print("  Flipping (W_j, b_j) -> (-W_j, -b_j) and outgoing weights")
print("  preserves the network function due to tanh(-z) = -tanh(z).")

In [ ]:
# Run full identifiability report on each activation
print("Running full identifiability analysis for each activation function...\n")

reports = {}
for act_name in ['sigmoid', 'tanh', 'relu']:
    torch.manual_seed(42)
    model = build_network(architecture, act_name)
    print(f"\n{'='*60}")
    report = identifiability_report(model, architecture, act_name)
    reports[act_name] = report

## 7. Network Architecture Visualization

Visualize the network structure using NetworkX to understand the topology and connectivity that Fefferman's conditions apply to.

In [ ]:
def draw_network_graph(architecture, title='Neural Network Architecture'):
    """Draw a neural network as a directed graph using NetworkX."""
    G = nx.DiGraph()
    pos = {}
    node_colors = []
    
    color_map = {0: '#3498db'}  # Input layer
    color_map[len(architecture) - 1] = '#e74c3c'  # Output layer
    
    # Add nodes
    for l, n_neurons in enumerate(architecture):
        for j in range(n_neurons):
            node_id = f'L{l}_N{j}'
            G.add_node(node_id)
            pos[node_id] = (l * 2, -(j - n_neurons / 2))
            
            if l == 0:
                node_colors.append('#3498db')
            elif l == len(architecture) - 1:
                node_colors.append('#e74c3c')
            else:
                node_colors.append('#2ecc71')
    
    # Add edges
    for l in range(len(architecture) - 1):
        for j in range(architecture[l]):
            for k in range(architecture[l + 1]):
                G.add_edge(f'L{l}_N{j}', f'L{l+1}_N{k}')
    
    fig, ax = plt.subplots(figsize=(14, 8))
    nx.draw(G, pos, ax=ax,
            node_color=node_colors,
            node_size=400,
            edge_color='gray',
            width=0.5,
            alpha=0.8,
            arrows=True,
            arrowsize=8,
            with_labels=False)
    
    # Layer labels
    for l, n in enumerate(architecture):
        label = 'Input' if l == 0 else ('Output' if l == len(architecture) - 1 else f'Hidden {l}')
        ax.text(l * 2, -(max(architecture) / 2 + 1), f'{label}\n({n} neurons)',
                ha='center', fontsize=11, fontweight='bold')
    
    patches = [
        mpatches.Patch(color='#3498db', label='Input'),
        mpatches.Patch(color='#2ecc71', label='Hidden'),
        mpatches.Patch(color='#e74c3c', label='Output'),
    ]
    ax.legend(handles=patches, loc='upper right', fontsize=11)
    ax.set_title(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Draw our test architecture
draw_network_graph([5, 10, 10, 1], title='Architecture [5, 10, 10, 1]')

In [ ]:
# Compare architectures of different sizes
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

test_architectures = [
    [3, 5, 1],
    [5, 10, 10, 1],
    [4, 8, 6, 4, 1],
]

for ax_idx, arch in enumerate(test_architectures):
    G = nx.DiGraph()
    pos = {}
    node_colors = []
    
    for l, n_neurons in enumerate(arch):
        for j in range(n_neurons):
            node_id = f'L{l}_N{j}'
            G.add_node(node_id)
            pos[node_id] = (l * 2, -(j - n_neurons / 2))
            if l == 0:
                node_colors.append('#3498db')
            elif l == len(arch) - 1:
                node_colors.append('#e74c3c')
            else:
                node_colors.append('#2ecc71')
    
    for l in range(len(arch) - 1):
        for j in range(arch[l]):
            for k in range(arch[l + 1]):
                G.add_edge(f'L{l}_N{j}', f'L{l+1}_N{k}')
    
    nx.draw(G, pos, ax=axes[ax_idx],
            node_color=node_colors, node_size=300,
            edge_color='gray', width=0.4, alpha=0.8,
            arrows=True, arrowsize=6, with_labels=False)
    axes[ax_idx].set_title(f'Architecture {arch}', fontsize=12, fontweight='bold')

plt.suptitle('Neural Network Architectures', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Summary of Identifiability Results

We compile all results into a summary table and discuss the implications.

In [ ]:
# Build summary table
print(f"{'='*80}")
print(f"IDENTIFIABILITY CONDITIONS SUMMARY")
print(f"Architecture: {architecture}")
print(f"{'='*80}")
print()
print(f"{'Condition':<25} {'Sigmoid':<15} {'Tanh':<15} {'ReLU':<15}")
print(f"{'-'*70}")

conditions = [
    ('No-Clones', 'no_clones', 'is_clone_free'),
    ('Non-Degeneracy', 'non_degeneracy', 'is_non_degenerate'),
    ('Self-Avoiding', 'self_avoiding', 'is_self_avoiding'),
    ('Activation Identifiable', None, 'activation_identifiable'),
    ('OVERALL', None, 'is_identifiable'),
]

for cond_name, key, subkey in conditions:
    row = f"{cond_name:<25}"
    for act in ['sigmoid', 'tanh', 'relu']:
        if key:
            val = reports[act][key][subkey]
        else:
            val = reports[act][subkey]
        status = 'PASS' if val else 'FAIL'
        row += f" {status:<15}"
    print(row)

print(f"\n{'='*80}")
print("\nEquivalence classes under identifiability:")
print("  Sigmoid : Identified up to neuron permutations")
print("  Tanh    : Identified up to neuron permutations AND sign flips (+/-)")
print("  ReLU    : NOT identifiable (fails Fefferman's genericity conditions)")

In [ ]:
# Visual summary bar chart
fig, ax = plt.subplots(figsize=(10, 5))

check_names = ['No-Clones', 'Non-Degenerate', 'Self-Avoiding', 'Activation OK', 'Overall']
x_pos = np.arange(len(check_names))
bar_width = 0.25

act_colors = {'sigmoid': '#3498db', 'tanh': '#e74c3c', 'relu': '#2ecc71'}

for i, (act, color) in enumerate(act_colors.items()):
    r = reports[act]
    values = [
        int(r['no_clones']['is_clone_free']),
        int(r['non_degeneracy']['is_non_degenerate']),
        int(r['self_avoiding']['is_self_avoiding']),
        int(r['activation_identifiable']),
        int(r['is_identifiable']),
    ]
    ax.bar(x_pos + i * bar_width, values, bar_width, label=act.capitalize(),
           color=color, alpha=0.8, edgecolor='black')

ax.set_xticks(x_pos + bar_width)
ax.set_xticklabels(check_names, rotation=15, ha='right')
ax.set_yticks([0, 1])
ax.set_yticklabels(['FAIL', 'PASS'])
ax.set_title('Identifiability Conditions by Activation Function', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. Conclusions

### Key Findings

1. **Random initialization** generally satisfies the no-clones, non-degeneracy, and self-avoiding conditions. This aligns with Fefferman's result that the set of non-identifiable parameters has **measure zero** in parameter space.

2. **Sigmoid and tanh** activations support identifiability because they are analytic. The equivalence classes are:
   - **Sigmoid**: Permutations only ($n_1! \times n_2! \times \ldots$ equivalent parameterizations)
   - **Tanh**: Permutations AND sign flips ($n_1! \cdot 2^{n_1} \times n_2! \cdot 2^{n_2} \times \ldots$)

3. **ReLU** fails because it is not analytic at zero. The positive homogeneity $\text{ReLU}(\alpha x) = \alpha \, \text{ReLU}(x)$ for $\alpha > 0$ creates a continuous family of equivalent parameterizations, not just discrete symmetries.

4. Deliberately injecting **clones** or **dead neurons** correctly triggers failure of the respective conditions, validating our implementation.

### Implications for Practice
- When interpretability matters, prefer **sigmoid or tanh** activations
- Always verify identifiability conditions before drawing conclusions about learned parameters
- Symmetry-breaking strategies (explored in Notebook 02) can further constrain the equivalence classes

---
*Next: See `02_Empirical_Analysis.ipynb` for isomorphism detection and symmetry-breaking experiments.*